In [10]:
from datetime import datetime, timedelta
from nba_api.stats.endpoints import scoreboardv2, boxscoretraditionalv2, boxscoretraditionalv3

In [19]:
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

In [27]:
def calculate_ttfl_score(df):
    df["TTFL"] = (
        df['FGM'] + df['FG3M'] + df['FTM'] + df['REB'] + df['AST'] + df['STL'] +
        df['BLK'] + df['PTS'] - df['TOV'] - (df['FGA'] - df['FGM']) -
        (df['FG3A'] - df['FG3M']) - (df['FTA'] - df['FTM'])
    )
    return df

In [ ]:
import re

def get_box_scores(game_date):
    
    scoreboard = scoreboardv2.ScoreboardV2(game_date=yesterday)
    games = scoreboard.get_normalized_dict()["GameHeader"]

    dfs = []
    for game in games:
        game_id = game["GAME_ID"]
        game_name = re.sub(r'^\d+\/([A-Z]{3})([A-Z]{3})$', r'\1 @ \2', game['GAMECODE'])

        boxscore = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id)
        player_stats = boxscore.player_stats.get_data_frame()

        df = player_stats[['teamTricode', 'nameI', 'minutes', 'points',
                        'fieldGoalsMade', 'fieldGoalsAttempted',
                        'threePointersMade', 'threePointersAttempted',
                        'freeThrowsMade', 'freeThrowsAttempted', 
                        'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers']]
        df.columns = ['Team', 'Name', 'MIN', 'PTS',
                        'FGM', 'FGA',
                        'FG3M', 'FG3A',
                        'FTM', 'FTA', 
                        'REB', 'AST', 'STL', 'BLK', 'TOV']
        df = calculate_ttfl_score(df)
        df = df[['Team', 'Name', 'TTFL', 'PTS', 'REB', 'AST', 'BLK' ,'TOV', 'MIN']]
        dfs.append((game_name, df))
    
    return(dfs)

In [37]:
a = get_box_scores(yesterday)

C:\Users\amaur\AppData\Local\Temp\ipykernel_12852\2835817324.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TTFL"] = (


In [38]:
a[0]

('LAC @ PHX',
    Team             Name  TTFL  PTS  REB  AST  BLK  TOV    MIN
 0   LAC         N. Batum    -3    0    2    1    0    0  18:24
 1   LAC     D. Jones Jr.    18   10    1    0    1    0  22:07
 2   LAC         I. Zubac    44   23   11    2    0    2  27:09
 3   LAC          B. Beal    -8    5    1    1    0    2  19:39
 4   LAC          K. Dunn    21    9    2    6    0    2  23:48
 5   LAC    B. Bogdanović    21   12    4    5    1    1  28:21
 6   LAC       J. Collins    15   13    4    1    1    4  23:03
 7   LAC          C. Paul     4    2    2    2    0    0   9:30
 8   LAC      C. Christie    30   17    2    3    0    0  25:07
 9   LAC         B. Lopez     5    6    3    0    0    2  17:49
 10  LAC         K. Brown     5    0    7    2    0    1  15:53
 11  LAC       J. Telfort    -2    1    1    1    0    1   6:33
 12  LAC  Y. Niederhäuser     8    4    1    0    1    0   2:37
 13  PHX         G. Allen    27   18    3    4    1    2  36:28
 14  PHX       R. O'Neale 